# MAISON KINON — Analyse du catalogue de parfums
**Projet :** Plateforme e-commerce de parfums de prestige  
**Auteur :** Kinon Ouattara  
**Objectif :** Collecter, nettoyer, analyser et modéliser les données du catalogue pour alimenter le dashboard admin et le moteur de recommandation.

**Source des données :** Fichier CSV `catalogue.csv` (export du catalogue produits).  
En production, cette source serait une table SQL (Supabase) ou un export automatique.

## 1. Import des bibliothèques et chargement des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    silhouette_score,
)
import joblib
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid', palette='Reds_r')
print('Bibliothèques chargées avec succès.')

In [ ]:
# ── Collecte automatisée : lecture du fichier CSV ──────────────────────────
# Le catalogue est stocké dans catalogue.csv (12 références).
# En production, on lirait directement depuis la base Supabase ou un export SQL.

df = pd.read_csv('catalogue.csv')

# Conversion du volume en entier
df['volume_ml'] = df['volume_ml'].astype(int)

print(f'Catalogue chargé depuis CSV : {len(df)} produits, {df.shape[1]} colonnes')
df.head()

## 2. Nettoyage et préparation des données

In [ ]:
print('=== Informations générales ===')
print(df.info())
print()
print('=== Valeurs manquantes ===')
print(df.isnull().sum())
print()
print('=== Doublons ===')
print(f'Nombre de doublons : {df.duplicated().sum()}')

In [ ]:
# Traitement des valeurs manquantes
# original_price = NaN signifie « pas de promotion » → on remplace par le prix normal
df['original_price'] = df['original_price'].fillna(df['price'])

# badge = NaN → 'Aucun'
df['badge'] = df['badge'].fillna('Aucun')

# Calcul de la remise en pourcentage
df['discount_pct'] = ((df['original_price'] - df['price']) / df['original_price'] * 100).round(1)

# Indicateur binaire : le produit est-il en promotion ?
df['is_promo'] = (df['discount_pct'] > 0).astype(int)

# Prix au millilitre (indicateur de valeur)
df['price_per_ml'] = (df['price'] / df['volume_ml']).round(3)

print(f'Dataset après nettoyage : {df.shape}')
df[['name', 'price', 'original_price', 'discount_pct', 'is_promo', 'price_per_ml']].head(6)

In [ ]:
# ── Export des données nettoyées vers CSV ──────────────────────────────────
# Le fichier nettoyé peut être réutilisé par l'application ou pour d'autres analyses.
df.to_csv('catalogue_nettoye.csv', index=False)
print('Données nettoyées exportées vers catalogue_nettoye.csv')

## 3. Statistiques descriptives et analyse

In [ ]:
print('=== Statistiques descriptives — prix ===')
print(df['price'].describe())
print()

print('=== Répartition par famille olfactive ===')
print(df['family'].value_counts())
print()

print('=== Répartition par genre ===')
print(df['gender'].value_counts())
print()

print('=== Prix moyen par famille ===')
print(df.groupby('family')['price'].mean().sort_values(ascending=False).round(2))

In [ ]:
# Tableau croisé dynamique : prix moyen par genre et famille
pivot = df.pivot_table(values='price', index='gender', columns='family', aggfunc='mean').round(0)
print('=== Prix moyen (€) par genre et famille olfactive ===')
print(pivot.fillna('-'))

## 4. Visualisations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('MAISON KINON — Analyse du catalogue', fontsize=14, fontweight='bold')

# 1. Répartition par famille olfactive
family_counts = df['family'].value_counts()
axes[0, 0].bar(family_counts.index, family_counts.values, color=['#c0392b','#e74c3c','#e8706b','#f0a09c','#f5c6c4'])
axes[0, 0].set_title('Nb de références par famille olfactive')
axes[0, 0].set_xlabel('Famille')
axes[0, 0].set_ylabel('Nb de produits')
axes[0, 0].tick_params(axis='x', rotation=30)

# 2. Distribution des prix
axes[0, 1].hist(df['price'], bins=6, color='#c0392b', edgecolor='white', alpha=0.85)
axes[0, 1].axvline(df['price'].mean(), color='black', linestyle='--', label=f"Moyenne : {df['price'].mean():.0f} €")
axes[0, 1].set_title('Distribution des prix (€)')
axes[0, 1].set_xlabel('Prix (€)')
axes[0, 1].set_ylabel('Fréquence')
axes[0, 1].legend()

# 3. Prix moyen par famille
avg_price = df.groupby('family')['price'].mean().sort_values()
axes[1, 0].barh(avg_price.index, avg_price.values, color='#e74c3c')
axes[1, 0].set_title('Prix moyen par famille olfactive (€)')
axes[1, 0].set_xlabel('Prix moyen (€)')

# 4. Répartition par genre (camembert)
gender_counts = df['gender'].value_counts()
axes[1, 1].pie(gender_counts.values, labels=gender_counts.index,
               autopct='%1.0f%%', colors=['#c0392b','#e8706b','#f5c6c4'],
               startangle=90, wedgeprops={'edgecolor':'white', 'linewidth':1.5})
axes[1, 1].set_title('Répartition par genre')

plt.tight_layout()
plt.savefig('catalogue_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graphiques enregistrés dans catalogue_analysis.png')

## 5. Machine Learning — Bloc 2
### 5a. Clustering K-Means (non supervisé)
**Problème ML :** Regrouper les parfums par profil de prix/volume pour identifier des segments naturels (entrée de gamme, milieu, premium).

**Évaluation :** Méthode du coude + score de silhouette (plus le score est proche de 1, meilleure est la séparation des clusters).

In [ ]:
# Préparation des features pour le clustering
features_cluster = df[['price', 'volume_ml', 'price_per_ml', 'discount_pct']].copy()
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(features_cluster)

# ── Méthode du coude + Score de silhouette ────────────────────────────────
inertias = []
silhouettes = []
k_range = range(2, 7)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(k_range, inertias, marker='o', color='#c0392b')
ax1.set_title('Méthode du coude — Inertie')
ax1.set_xlabel('Nombre de clusters k')
ax1.set_ylabel('Inertie')

ax2.plot(k_range, silhouettes, marker='s', color='#e74c3c')
ax2.set_title('Score de silhouette')
ax2.set_xlabel('Nombre de clusters k')
ax2.set_ylabel('Score de silhouette')

plt.tight_layout()
plt.savefig('kmeans_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print('Scores de silhouette par k :')
for k, s in zip(k_range, silhouettes):
    print(f'  k={k} → silhouette={s:.3f}')

In [ ]:
# Entraînement avec k=3 (3 segments : entrée / milieu / premium)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['segment'] = kmeans.fit_predict(X_scaled)

# Renommage des segments selon le prix moyen
segment_labels = df.groupby('segment')['price'].mean().sort_values()
label_map = {segment_labels.index[0]: 'Entrée de gamme',
             segment_labels.index[1]: 'Milieu de gamme',
             segment_labels.index[2]: 'Premium'}
df['segment_label'] = df['segment'].map(label_map)

# Score de silhouette final pour k=3
sil_final = silhouette_score(X_scaled, df['segment'])
print(f'=== Segments identifiés par K-Means (silhouette = {sil_final:.3f}) ===')
print(df.groupby('segment_label')[['price', 'volume_ml', 'price_per_ml']].mean().round(2))
print()
print('=== Répartition des produits par segment ===')
print(df[['name', 'price', 'segment_label']].sort_values('price'))

### 5b. Classification supervisée — Comparaison de modèles
**Problème ML :** Prédire le genre cible d'un parfum (Femme / Homme) à partir de ses caractéristiques (prix, volume, famille, remise).

**Démarche :**
1. Sélection des variables et encodage
2. Séparation train/test (70/30)
3. Entraînement de **deux modèles** : Random Forest et Régression Logistique
4. Évaluation avec accuracy, précision, rappel, F1 et matrice de confusion
5. Validation croisée (5-fold)
6. Sauvegarde du meilleur modèle

In [ ]:
# Encodage des variables catégorielles
df_ml = df[df['gender'].isin(['Femme', 'Homme'])].copy()  # on exclut Unisexe (1 produit)

le_family = LabelEncoder()
df_ml['family_enc'] = le_family.fit_transform(df_ml['family'])

le_gender = LabelEncoder()
df_ml['gender_enc'] = le_gender.fit_transform(df_ml['gender'])  # Femme=0, Homme=1

# Features et cible
feature_cols = ['price', 'volume_ml', 'price_per_ml', 'discount_pct', 'family_enc']
X = df_ml[feature_cols]
y = df_ml['gender_enc']

print(f'Dataset ML : {len(X)} échantillons, {len(feature_cols)} features')
print(f'Classes : {le_gender.classes_}  |  Distribution : {y.value_counts().to_dict()}')

In [ ]:
# Séparation train/test (70/30 — petit dataset, stratification pour équilibrer)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y,
)
print(f'Train : {len(X_train)} | Test : {len(X_test)}')

# ── Modèle 1 : Random Forest ──────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)

# ── Modèle 2 : Régression Logistique ──────────────────────────────────────
lr = LogisticRegression(max_iter=500, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)

# ── Comparaison des deux modèles ──────────────────────────────────────────
print('=' * 50)
print('  COMPARAISON DES MODÈLES')
print('=' * 50)
print(f'  Random Forest         → Accuracy = {acc_rf:.2%}')
print(f'  Régression Logistique → Accuracy = {acc_lr:.2%}')
print()
print('--- Random Forest : rapport de classification ---')
print(classification_report(y_test, y_pred_rf, target_names=le_gender.classes_))
print('--- Régression Logistique : rapport de classification ---')
print(classification_report(y_test, y_pred_lr, target_names=le_gender.classes_))

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred_rf)
print('Matrice de confusion (Random Forest) :')
print(cm)

In [ ]:
# ── Validation croisée 5-fold ──────────────────────────────────────────────
# La validation croisée évalue la robustesse du modèle en divisant les données
# en 5 plis et en moyennant les scores.

cv_rf = cross_val_score(rf, X, y, cv=5, scoring='accuracy')
cv_lr = cross_val_score(lr, X, y, cv=5, scoring='accuracy')

print('=== Validation croisée 5-fold ===')
print(f'  Random Forest         : {cv_rf.mean():.2%} (±{cv_rf.std():.2%})  → folds: {cv_rf.round(2)}')
print(f'  Régression Logistique : {cv_lr.mean():.2%} (±{cv_lr.std():.2%})  → folds: {cv_lr.round(2)}')

In [ ]:
# Importance des features (Random Forest)
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)

plt.figure(figsize=(7, 4))
importances.plot(kind='barh', color='#c0392b')
plt.title('Importance des features — Random Forest')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Conclusion : la famille olfactive et le prix sont les variables les plus discriminantes pour prédire le genre cible.')

In [ ]:
# ── Export du meilleur modèle ──────────────────────────────────────────────
# On sauvegarde le modèle et les encodeurs pour une réutilisation future
# (déploiement dans l'application, inférence sur de nouvelles données).

best_model = rf if acc_rf >= acc_lr else lr
best_name = 'RandomForest' if acc_rf >= acc_lr else 'LogisticRegression'

joblib.dump(best_model, 'modele_genre.pkl')
joblib.dump(kmeans, 'modele_kmeans.pkl')
joblib.dump(le_family, 'encodeur_famille.pkl')
joblib.dump(le_gender, 'encodeur_genre.pkl')
joblib.dump(scaler, 'scaler.pkl')

print(f'Meilleur modèle : {best_name} (accuracy = {max(acc_rf, acc_lr):.2%})')
print('Modèles exportés : modele_genre.pkl, modele_kmeans.pkl, encodeurs, scaler')

# ── Démonstration : prédiction sur un nouveau produit ───────────────────────
nouveau_produit = pd.DataFrame({
    'price': [120], 'volume_ml': [100], 'price_per_ml': [1.20],
    'discount_pct': [0], 'family_enc': [le_family.transform(['Floral'])[0]],
})
prediction = best_model.predict(nouveau_produit)
genre_predit = le_gender.inverse_transform(prediction)[0]
print(f'\nPrédiction pour un parfum Floral 100ml à 120€ → Genre prédit : {genre_predit}')

### 5c. Moteur de recommandation — Similarité cosinus
**Objectif :** Pour un produit donné, trouver les N parfums les plus similaires.  
C'est l'algorithme qui alimente la section **"Vous pourriez aussi aimer"** de l'application.

In [ ]:
# Encodage de toutes les variables pour la similarité
df_reco = df.copy()
le_f  = LabelEncoder(); df_reco['family_enc']  = le_f.fit_transform(df_reco['family'])
le_g  = LabelEncoder(); df_reco['gender_enc']  = le_g.fit_transform(df_reco['gender'])

reco_features = ['price', 'volume_ml', 'price_per_ml', 'family_enc', 'gender_enc']
X_reco = MinMaxScaler().fit_transform(df_reco[reco_features])

# Matrice de similarité cosinus
sim_matrix = cosine_similarity(X_reco)

def get_recommendations(product_name: str, n: int = 4) -> pd.DataFrame:
    """Retourne les n parfums les plus similaires à product_name."""
    idx = df_reco[df_reco['name'].str.contains(product_name, case=False)].index[0]
    scores = list(enumerate(sim_matrix[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    top_idx = [i for i, _ in scores[1:n+1]]  # exclut lui-même
    result = df_reco.iloc[top_idx][['name', 'brand', 'price', 'family', 'gender']].copy()
    result['score_similarité'] = [round(scores[i][1], 3) for i in range(1, n+1)]
    return result

print('=== Recommandations pour "Sauvage" (Dior) ===')
print(get_recommendations('Sauvage', n=4))
print()
print('=== Recommandations pour "N°5" (Chanel) ===')
print(get_recommendations('N°5', n=4))

## 6. Synthèse et conclusions

In [ ]:
print('=' * 55)
print('  MAISON KINON — Synthèse de l\'analyse')
print('=' * 55)
print(f'  Nb de produits analysés     : {len(df)}')
print(f'  Prix moyen                  : {df["price"].mean():.0f} €')
print(f'  Prix médian                 : {df["price"].median():.0f} €')
print(f'  Produits en promotion       : {df["is_promo"].sum()} / {len(df)}')
print(f'  Famille la plus représentée : {df["family"].mode()[0]}')
print(f'  Genre dominant              : {df["gender"].value_counts().index[0]}')
print()
print('  Machine Learning :')
print(f'  - Clustering K-Means (k=3) : silhouette = {sil_final:.3f}')
print(f'  - Random Forest  : accuracy = {acc_rf:.2%} (CV5 = {cv_rf.mean():.2%})')
print(f'  - Régr. Logistique: accuracy = {acc_lr:.2%} (CV5 = {cv_lr.mean():.2%})')
print(f'  - Meilleur modèle : {best_name}')
print(f'  - Reco cosinus   : 5 features (prix, volume, prix/ml, famille, genre)')
print()
print('  Fichiers exportés :')
print('  - catalogue_nettoye.csv (données nettoyées)')
print('  - modele_genre.pkl, modele_kmeans.pkl (modèles sauvegardés)')
print('  - catalogue_analysis.png, kmeans_evaluation.png (graphiques)')
print('=' * 55)